# Demo RAG con Azure AI Search y Microsoft Agent Framework

Este notebook implementa una demo **RAG clásica y explicable** usando servicios nativos de Azure para la ingestión:

1. Descarga un subconjunto libre de documentos de **misiones a Marte de la NASA**.
2. Publica los PDF en Azure Blob Storage con metadata de fuente.
3. Usa Azure AI Search para extraer texto, dividirlo con **Text Split Skill** y generar embeddings con **AzureOpenAIEmbeddingSkill**.
4. Crea y llena un índice vectorial mediante un indexer de Azure AI Search.
5. Expone la búsqueda híbrida como una herramienta de Microsoft Agent Framework.
6. Ejecuta preguntas con respuestas fundamentadas y referencias a las fuentes.

## Arquitectura

```text
Ingestión
PDFs locales
  │
  ▼
Azure Blob Storage
  │
  ▼
Azure AI Search Indexer
  │ extrae texto del PDF
  ├── Text Split Skill: chunks
  └── AzureOpenAIEmbeddingSkill: vectores
  ▼
Índice vectorial + Semantic Ranker

Consulta
Usuario
  │
  ▼
Microsoft Agent Framework
  │ decide llamar a la herramienta
  ▼
buscar_en_base_documental()
  │
  ├── embedding de la consulta
  ▼
Azure AI Search
  │ búsqueda híbrida: texto + vector + rerank
  ▼
fragmentos relevantes
  │
  ▼
Modelo de chat en Azure OpenAI / Microsoft Foundry
  │
  ▼
Respuesta con fuentes
```

> La teoría de chunking se puede explicar en la presentación; el código usa la mayor cantidad posible de capacidades administradas de Azure para la demo.


## 0. Preparación en Azure

### Recursos mínimos

1. **Grupo de recursos**.
2. **Azure OpenAI in Microsoft Foundry Models** o un **proyecto de Microsoft Foundry** con endpoint compatible con Azure OpenAI.
3. Dos deployments:
   - Chat: `gpt-4.1-mini`, `gpt-4o-mini` u otro modelo compatible con Responses API.
   - Embeddings: `text-embedding-3-small`.
4. **Azure AI Search**:
   - `Basic` o superior es lo más cómodo para una demo estable con indexers, skillsets y semantic ranker.
5. **Azure Blob Storage**:
   - Un contenedor para los PDF que serán procesados por el indexer.

### Roles RBAC para tu usuario

En el recurso de modelos/OpenAI:

- `Cognitive Services OpenAI User`

En Azure AI Search:

- `Search Service Contributor`
- `Search Index Data Contributor`
- `Search Index Data Reader`

Si el skill de embeddings usa identidad administrada en vez de `AZURE_OPENAI_API_KEY`, habilita la identidad administrada del servicio de Azure AI Search y asígnale `Cognitive Services OpenAI User` en el recurso de Azure OpenAI.

Después, autentícate localmente:

```bash
az login
az account set --subscription "<SUBSCRIPTION_ID_O_NOMBRE>"
```

### Variables de entorno

Crea un archivo `.env` junto al notebook:

```dotenv
AZURE_OPENAI_ENDPOINT=https://<recurso>.openai.azure.com
AZURE_OPENAI_API_VERSION=2025-04-01-preview
AZURE_OPENAI_CHAT_MODEL=<nombre-del-deployment-chat>
AZURE_OPENAI_EMBEDDING_MODEL=<nombre-del-deployment-embedding>
AZURE_OPENAI_EMBEDDING_MODEL_NAME=text-embedding-3-small
# Opcional si Azure AI Search no usará identidad administrada para embeddings
# AZURE_OPENAI_API_KEY=<api-key>

AZURE_SEARCH_ENDPOINT=https://<servicio>.search.windows.net
AZURE_SEARCH_INDEX=rag-nasa-agent-demo
AZURE_SEARCH_API_VERSION=2025-09-01
EMBEDDING_DIMENSIONS=1536

AZURE_STORAGE_CONNECTION_STRING=DefaultEndpointsProtocol=https;AccountName=<storage-account>;AccountKey=<storage-key>;EndpointSuffix=core.windows.net
AZURE_STORAGE_CONTAINER=mars-rag
AZURE_STORAGE_BLOB_PREFIX=mars-rag

AZURE_SEARCH_TEXT_SPLIT_MODE=pages
AZURE_SEARCH_TEXT_SPLIT_UNIT=characters
AZURE_SEARCH_TEXT_SPLIT_MAX_LENGTH=3500
AZURE_SEARCH_TEXT_SPLIT_OVERLAP=500
AZURE_SEARCH_TEXT_SPLIT_LANGUAGE=en
RESET_SEARCH_ASSETS=true
WAIT_FOR_INDEXER=true
MAX_PDFS=10
```

Los valores de `*_MODEL` son **nombres de deployments**. `AZURE_OPENAI_EMBEDDING_MODEL_NAME` es el nombre base del modelo que espera el skill de Azure AI Search.


## 1. Instalar dependencias

Se fija Microsoft Agent Framework a una versión conocida para que la demo sea reproducible.
Después de instalar, reinicia el kernel si Jupyter lo solicita.


In [ ]:
%pip uninstall -y agent-framework agent-framework-azure-ai-search

%pip install --upgrade pip

%pip install   "agent-framework-core==1.8.1"   "agent-framework-openai==1.8.1"   "azure-search-documents==12.0.0"   "azure-identity>=1.17,<2"   "azure-storage-blob>=12.20,<13"   "python-dotenv>=1.0,<2"   "httpx>=0.27,<1"   "tenacity>=9,<10"   "pydantic>=2.9,<3"


## 2. Cargar configuración y autenticar

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()

required = [
    "AZURE_OPENAI_ENDPOINT",
    "AZURE_OPENAI_CHAT_MODEL",
    "AZURE_OPENAI_EMBEDDING_MODEL",
    "AZURE_SEARCH_ENDPOINT",
    "AZURE_STORAGE_CONTAINER",
]

missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(
        "Faltan variables de entorno: "
        + ", ".join(missing)
        + ". Crea el archivo .env antes de continuar."
    )

AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION", "2025-04-01-preview")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
CHAT_MODEL = os.environ["AZURE_OPENAI_CHAT_MODEL"]
EMBEDDING_MODEL = os.environ["AZURE_OPENAI_EMBEDDING_MODEL"]
EMBEDDING_MODEL_NAME = os.getenv("AZURE_OPENAI_EMBEDDING_MODEL_NAME", EMBEDDING_MODEL)

AZURE_SEARCH_ENDPOINT = os.environ["AZURE_SEARCH_ENDPOINT"].rstrip("/")
AZURE_SEARCH_INDEX = os.getenv("AZURE_SEARCH_INDEX", "rag-nasa-agent-demo")
AZURE_SEARCH_API_VERSION = os.getenv("AZURE_SEARCH_API_VERSION", "2025-09-01")

AZURE_STORAGE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
AZURE_SUBSCRIPTION_ID = os.getenv("AZURE_SUBSCRIPTION_ID")
AZURE_RESOURCE_GROUP = os.getenv("AZURE_RESOURCE_GROUP")
AZURE_STORAGE_ACCOUNT_NAME = os.getenv("AZURE_STORAGE_ACCOUNT_NAME")
AZURE_STORAGE_ACCOUNT_URL = os.getenv("AZURE_STORAGE_ACCOUNT_URL")
AZURE_STORAGE_RESOURCE_ID = os.getenv("AZURE_STORAGE_RESOURCE_ID")
AZURE_STORAGE_CONTAINER = os.environ["AZURE_STORAGE_CONTAINER"]
AZURE_STORAGE_BLOB_PREFIX = os.getenv("AZURE_STORAGE_BLOB_PREFIX", "mars-rag").strip("/")


def is_configured(value: str | None) -> bool:
    return bool(value and "<" not in value and "..." not in value)

TEXT_SPLIT_MODE = os.getenv("AZURE_SEARCH_TEXT_SPLIT_MODE", "pages")
TEXT_SPLIT_UNIT = os.getenv("AZURE_SEARCH_TEXT_SPLIT_UNIT", "characters")
TEXT_SPLIT_MAX_LENGTH = int(os.getenv("AZURE_SEARCH_TEXT_SPLIT_MAX_LENGTH", "3500"))
TEXT_SPLIT_OVERLAP = int(os.getenv("AZURE_SEARCH_TEXT_SPLIT_OVERLAP", "500"))
TEXT_SPLIT_LANGUAGE = os.getenv("AZURE_SEARCH_TEXT_SPLIT_LANGUAGE", "en")

RESET_SEARCH_ASSETS = os.getenv("RESET_SEARCH_ASSETS", "true").lower() == "true"
WAIT_FOR_INDEXER = os.getenv("WAIT_FOR_INDEXER", "true").lower() == "true"
EMBEDDING_DIMENSIONS = int(os.getenv("EMBEDDING_DIMENSIONS", "1536"))
MAX_PDFS = int(os.getenv("MAX_PDFS", "10"))

DATA_DIR = Path("data/mars-rag")
DATA_DIR.mkdir(parents=True, exist_ok=True)

credential = AzureCliCredential()

print("Configuración cargada")
print(f"  Chat deployment:       {CHAT_MODEL}")
print(f"  Embedding deployment:  {EMBEDDING_MODEL}")
print(f"  Search index:          {AZURE_SEARCH_INDEX}")
print(f"  Storage account:       {AZURE_STORAGE_ACCOUNT_NAME or '(connection string)'}")
print(f"  Storage container:     {AZURE_STORAGE_CONTAINER}")
print(f"  Blob prefix:           {AZURE_STORAGE_BLOB_PREFIX or '(raíz del contenedor)'}")
print(f"  PDFs a descargar:      {MAX_PDFS}")


## 3. Descargar el dataset

Para este caso de uso enfocado en misiones a Marte, hemos definido una lista específica de documentos oficiales de la NASA (como los Press Kits de Perseverance, Curiosity, InSight, entre otros).

El notebook descarga estos PDFs directamente desde los repositorios públicos de la NASA (JPL y NTRS) y los almacena en la carpeta local `data/mars-rag`. Además, conserva la URL pública de origen de cada documento para poder inyectarla posteriormente en las referencias (citas) de las respuestas del agente.


In [22]:
from pathlib import Path
import re

import httpx

DATA_DIR = Path("data/mars-rag")
DATA_DIR.mkdir(parents=True, exist_ok=True)

MARS_DOCUMENTS = [
    {
        "title": "Mars 2020 Perseverance Launch Press Kit",
        "filename": "mars-2020-launch-press-kit.pdf",
        "url": (
            "https://www.jpl.nasa.gov/news/press_kits/"
            "mars_2020/download/mars_2020_launch_press_kit.pdf"
        ),
        "category": "robotic-mission",
        "mission": "Perseverance",
    },
    {
        "title": "Mars 2020 Perseverance Landing Press Kit",
        "filename": "mars-2020-landing-press-kit.pdf",
        "url": (
            "https://www.jpl.nasa.gov/news/press_kits/"
            "mars_2020/download/mars_2020_landing_press_kit.pdf"
        ),
        "category": "robotic-mission",
        "mission": "Perseverance",
    },
    {
        "title": "Mars Science Laboratory Curiosity Landing Press Kit",
        "filename": "curiosity-landing-press-kit.pdf",
        "url": (
            "https://www.jpl.nasa.gov/news/press_kits/"
            "MSLLanding.pdf"
        ),
        "category": "robotic-mission",
        "mission": "Curiosity",
    },
    {
        "title": "Mars Reconnaissance Orbiter Launch Press Kit",
        "filename": "mro-launch-press-kit.pdf",
        "url": (
            "https://www.jpl.nasa.gov/news/press_kits/"
            "mro-launch.pdf"
        ),
        "category": "robotic-mission",
        "mission": "Mars Reconnaissance Orbiter",
    },
    {
        "title": "MAVEN Press Kit",
        "filename": "maven-press-kit.pdf",
        "url": (
            "https://www.nasa.gov/wp-content/uploads/"
            "2015/03/maven_presskit_final2.pdf"
        ),
        "category": "robotic-mission",
        "mission": "MAVEN",
    },
    {
        "title": "Mars InSight Landing Press Kit",
        "filename": "insight-landing-press-kit.pdf",
        "url": (
            "https://www.jpl.nasa.gov/news/press_kits/"
            "insight/landing/download/"
            "mars_insight_landing_presskit.pdf"
        ),
        "category": "robotic-mission",
        "mission": "InSight",
    },
    {
        "title": "Phoenix Mars Lander Launch Press Kit",
        "filename": "phoenix-launch-press-kit.pdf",
        "url": (
            "https://www.jpl.nasa.gov/news/press_kits/"
            "phoenix-launch-presskit.pdf"
        ),
        "category": "robotic-mission",
        "mission": "Phoenix",
    },
    {
        "title": "Human Exploration of Mars Design Reference Architecture 5.0",
        "filename": "human-mars-dra-5.pdf",
        "url": (
            "https://ntrs.nasa.gov/api/citations/"
            "20090040343/downloads/20090040343.pdf"
        ),
        "category": "human-exploration",
        "mission": "Human Mars Architecture",
    },
    {
        "title": "NASA Human Missions to Mars",
        "filename": "nasa-human-missions-to-mars.pdf",
        "url": (
            "https://ntrs.nasa.gov/api/citations/"
            "20160013646/downloads/20160013646.pdf"
        ),
        "category": "human-exploration",
        "mission": "Human Mars Architecture",
    },
    {
        "title": "Human Exploration of Mars Preliminary Crew Tasks",
        "filename": "human-mars-crew-tasks.pdf",
        "url": (
            "https://ntrs.nasa.gov/api/citations/"
            "20190001401/downloads/20190001401.pdf"
        ),
        "category": "human-exploration",
        "mission": "Human Mars Operations",
    },
]

MAX_PDFS = min(MAX_PDFS, len(MARS_DOCUMENTS))

headers = {
    "User-Agent": "azure-rag-agent-framework-demo",
}

downloaded_files = []

with httpx.Client(
    follow_redirects=True,
    timeout=180.0,
    headers=headers,
) as client:
    for document in MARS_DOCUMENTS[:MAX_PDFS]:
        local_path = DATA_DIR / document["filename"]

        if not local_path.exists():
            print("Descargando:", document["title"])

            response = client.get(document["url"])
            response.raise_for_status()

            content_type = response.headers.get(
                "content-type",
                "",
            ).lower()

            if (
                not response.content.startswith(b"%PDF")
                and "application/pdf" not in content_type
            ):
                raise RuntimeError(
                    f'La descarga de "{document["title"]}" '
                    "no parece ser un PDF."
                )

            local_path.write_bytes(response.content)

        downloaded_files.append(
            {
                "path": local_path,
                "source_url": document["url"],
                "name": document["filename"],
                "title": document["title"],
                "category": document["category"],
                "mission": document["mission"],
                "source_page": None,
            }
        )

print(f"\nPDFs disponibles: {len(downloaded_files)}")

for document in downloaded_files:
    size_mb = document["path"].stat().st_size / 1024 / 1024

    print(
        f'- {document["title"]} '
        f'[{document["category"]}] '
        f'({size_mb:.1f} MB)'
    )


PDFs disponibles: 10
- Mars 2020 Perseverance Launch Press Kit [robotic-mission] (15.4 MB)
- Mars 2020 Perseverance Landing Press Kit [robotic-mission] (10.7 MB)
- Mars Science Laboratory Curiosity Landing Press Kit [robotic-mission] (5.0 MB)
- Mars Reconnaissance Orbiter Launch Press Kit [robotic-mission] (0.2 MB)
- MAVEN Press Kit [robotic-mission] (2.1 MB)
- Mars InSight Landing Press Kit [robotic-mission] (13.0 MB)
- Phoenix Mars Lander Launch Press Kit [robotic-mission] (6.2 MB)
- Human Exploration of Mars Design Reference Architecture 5.0 [human-exploration] (7.8 MB)
- NASA Human Missions to Mars [human-exploration] (0.9 MB)
- Human Exploration of Mars Preliminary Crew Tasks [human-exploration] (1.0 MB)


## 4. Publicar PDFs en Azure Blob Storage

Azure AI Search puede leer directamente desde Blob Storage. En esta celda subimos los documentos descargados y agregamos metadata que luego se proyecta hacia el índice de chunks.


In [ ]:
from azure.core.exceptions import ResourceExistsError
from azure.storage.blob import BlobServiceClient, ContentSettings


def create_blob_service() -> BlobServiceClient:
    if is_configured(AZURE_STORAGE_CONNECTION_STRING):
        print("Autenticando Storage con connection string.")
        return BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)

    account_url = AZURE_STORAGE_ACCOUNT_URL
    if not is_configured(account_url) and is_configured(AZURE_STORAGE_ACCOUNT_NAME):
        account_url = f"https://{AZURE_STORAGE_ACCOUNT_NAME}.blob.core.windows.net"

    if is_configured(account_url):
        print("Autenticando Storage con Azure AD.")
        return BlobServiceClient(account_url=account_url, credential=credential)

    raise RuntimeError(
        "Falta configurar Storage. Define AZURE_STORAGE_CONNECTION_STRING o "
        "AZURE_STORAGE_ACCOUNT_NAME en el .env."
    )


def blob_name_for(filename: str) -> str:
    if not AZURE_STORAGE_BLOB_PREFIX:
        return filename
    return f"{AZURE_STORAGE_BLOB_PREFIX}/{filename}"


def blob_metadata(document: dict) -> dict:
    return {
        "title": document["title"],
        "source_url": document["source_url"],
        "category": document["category"],
        "mission": document["mission"],
    }


blob_service = create_blob_service()
container_client = blob_service.get_container_client(AZURE_STORAGE_CONTAINER)

try:
    container_client.create_container()
    print("Contenedor creado:", AZURE_STORAGE_CONTAINER)
except ResourceExistsError:
    print("Usando contenedor existente:", AZURE_STORAGE_CONTAINER)

blob_manifest = []

for document in downloaded_files:
    blob_name = blob_name_for(document["name"])
    blob_client = container_client.get_blob_client(blob_name)

    print(f"Subiendo: {document['title']} -> {blob_name}")
    with document["path"].open("rb") as pdf_file:
        blob_client.upload_blob(
            pdf_file,
            overwrite=True,
            metadata=blob_metadata(document),
            content_settings=ContentSettings(content_type="application/pdf"),
        )

    blob_manifest.append(
        {
            "title": document["title"],
            "blob_name": blob_name,
            "source_url": document["source_url"],
            "category": document["category"],
            "mission": document["mission"],
        }
    )

print("PDFs publicados en Blob Storage:", len(blob_manifest))


## 5. Configurar embeddings de consulta con Microsoft Agent Framework

Los embeddings de los documentos se generan dentro de Azure AI Search mediante `AzureOpenAIEmbeddingSkill`. En el notebook conservamos un cliente de embeddings solo para vectorizar la pregunta del usuario al consultar.


In [ ]:
from agent_framework.openai import OpenAIEmbeddingClient
from tenacity import retry, stop_after_attempt, wait_exponential

embedding_client = OpenAIEmbeddingClient(
    model=EMBEDDING_MODEL,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=AZURE_OPENAI_API_VERSION,
    credential=credential,
)


@retry(
    wait=wait_exponential(multiplier=1, min=2, max=20),
    stop=stop_after_attempt(5),
    reraise=True,
)
async def embed_texts(texts: list[str]) -> list[list[float]]:
    generated = await embedding_client.get_embeddings(texts)
    vectors = [list(item.vector) for item in generated]

    for vector in vectors:
        if len(vector) != EMBEDDING_DIMENSIONS:
            raise ValueError(
                f"El embedding tiene {len(vector)} dimensiones, "
                f"pero el índice está configurado para {EMBEDDING_DIMENSIONS}."
            )

    return vectors


print("Cliente de embeddings listo para vectorizar consultas.")


## 6. Crear índice, data source, skillset e indexer en Azure AI Search

Esta es la parte clave de la demo: el índice se llena con un pipeline administrado de Azure AI Search. El `Text Split Skill` crea los chunks, el `AzureOpenAIEmbeddingSkill` genera el vector de cada chunk y `indexProjections` escribe un documento por fragmento.


In [ ]:
import httpx

DATA_SOURCE_NAME = f"{AZURE_SEARCH_INDEX}-blob-datasource"
SKILLSET_NAME = f"{AZURE_SEARCH_INDEX}-textsplit-skillset"
INDEXER_NAME = f"{AZURE_SEARCH_INDEX}-blob-indexer"


def search_url(path: str) -> str:
    separator = "&" if "?" in path else "?"
    return f"{AZURE_SEARCH_ENDPOINT}{path}{separator}api-version={AZURE_SEARCH_API_VERSION}"


def resource_path(collection: str, name: str) -> str:
    return f"/{collection}('{name}')"


def auth_headers() -> dict[str, str]:
    token = credential.get_token("https://search.azure.com/.default").token
    return {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json",
    }


def request_search(
    client: httpx.Client,
    method: str,
    path: str,
    *,
    json_body: dict | None = None,
    expected: tuple[int, ...] = (200, 201, 202, 204),
    ignore_404: bool = False,
) -> httpx.Response | None:
    response = client.request(method, search_url(path), json=json_body)
    if ignore_404 and response.status_code == 404:
        return None
    if response.status_code not in expected:
        raise RuntimeError(
            f"{method} {path} falló con HTTP {response.status_code}:\n{response.text}"
        )
    return response


def index_definition() -> dict:
    return {
        "name": AZURE_SEARCH_INDEX,
        "fields": [
            {
                "name": "id",
                "type": "Edm.String",
                "key": True,
                "searchable": True,
                "filterable": True,
                "retrievable": True,
                "analyzer": "keyword",
            },
            {"name": "parent_id", "type": "Edm.String", "filterable": True},
            {
                "name": "title",
                "type": "Edm.String",
                "searchable": True,
                "filterable": True,
                "sortable": True,
                "retrievable": True,
            },
            {
                "name": "content",
                "type": "Edm.String",
                "searchable": True,
                "retrievable": True,
            },
            {
                "name": "source_url",
                "type": "Edm.String",
                "filterable": True,
                "retrievable": True,
            },
            {
                "name": "page_number",
                "type": "Edm.Int32",
                "filterable": True,
                "sortable": True,
                "retrievable": True,
            },
            {
                "name": "category",
                "type": "Edm.String",
                "searchable": True,
                "filterable": True,
                "facetable": True,
                "retrievable": True,
            },
            {
                "name": "mission",
                "type": "Edm.String",
                "searchable": True,
                "filterable": True,
                "facetable": True,
                "retrievable": True,
            },
            {
                "name": "content_vector",
                "type": "Collection(Edm.Single)",
                "searchable": True,
                "retrievable": False,
                "dimensions": EMBEDDING_DIMENSIONS,
                "vectorSearchProfile": "rag-vector-profile",
            },
        ],
        "vectorSearch": {
            "algorithms": [
                {
                    "name": "rag-hnsw",
                    "kind": "hnsw",
                    "hnswParameters": {"metric": "cosine"},
                }
            ],
            "profiles": [
                {
                    "name": "rag-vector-profile",
                    "algorithm": "rag-hnsw",
                }
            ],
        },
        "semantic": {
            "configurations": [
                {
                    "name": "rag-semantic-config",
                    "prioritizedFields": {
                        "titleField": {"fieldName": "title"},
                        "prioritizedContentFields": [{"fieldName": "content"}],
                    },
                }
            ]
        },
    }


def storage_resource_id() -> str | None:
    if is_configured(AZURE_STORAGE_RESOURCE_ID):
        return AZURE_STORAGE_RESOURCE_ID

    if (
        is_configured(AZURE_SUBSCRIPTION_ID)
        and is_configured(AZURE_RESOURCE_GROUP)
        and is_configured(AZURE_STORAGE_ACCOUNT_NAME)
    ):
        return (
            f"/subscriptions/{AZURE_SUBSCRIPTION_ID}"
            f"/resourceGroups/{AZURE_RESOURCE_GROUP}"
            f"/providers/Microsoft.Storage/storageAccounts/{AZURE_STORAGE_ACCOUNT_NAME}"
        )

    return None


def storage_data_source_connection_string() -> str:
    if is_configured(AZURE_STORAGE_CONNECTION_STRING):
        return AZURE_STORAGE_CONNECTION_STRING

    resource_id = storage_resource_id()
    if resource_id:
        return f"ResourceId={resource_id};"

    raise RuntimeError(
        "Falta configurar el origen de Blob Storage para Azure AI Search. Define "
        "AZURE_STORAGE_CONNECTION_STRING o AZURE_STORAGE_RESOURCE_ID."
    )


def data_source_definition() -> dict:
    container = {"name": AZURE_STORAGE_CONTAINER}
    if AZURE_STORAGE_BLOB_PREFIX:
        container["query"] = AZURE_STORAGE_BLOB_PREFIX

    return {
        "name": DATA_SOURCE_NAME,
        "type": "azureblob",
        "credentials": {"connectionString": storage_data_source_connection_string()},
        "container": container,
    }


def skillset_definition() -> dict:
    split_skill = {
        "@odata.type": "#Microsoft.Skills.Text.SplitSkill",
        "name": "split-content",
        "description": "Divide el contenido del PDF en fragmentos listos para RAG.",
        "context": "/document",
        "textSplitMode": TEXT_SPLIT_MODE,
        "maximumPageLength": TEXT_SPLIT_MAX_LENGTH,
        "pageOverlapLength": TEXT_SPLIT_OVERLAP,
        "defaultLanguageCode": TEXT_SPLIT_LANGUAGE,
        "inputs": [{"name": "text", "source": "/document/content"}],
        "outputs": [
            {"name": "textItems", "targetName": "pages"},
        ],
    }

    if TEXT_SPLIT_UNIT != "characters":
        if "preview" not in AZURE_SEARCH_API_VERSION:
            raise RuntimeError(
                "AZURE_SEARCH_TEXT_SPLIT_UNIT distinto de 'characters' requiere una "
                "API preview de Azure AI Search. Para la demo estable usa "
                "AZURE_SEARCH_TEXT_SPLIT_UNIT=characters o elimina esa variable."
            )
        split_skill["unit"] = TEXT_SPLIT_UNIT

    embedding_skill = {
        "@odata.type": "#Microsoft.Skills.Text.AzureOpenAIEmbeddingSkill",
        "name": "embed-pages",
        "description": "Genera embeddings para cada chunk creado por Text Split.",
        "context": "/document/pages/*",
        "resourceUri": AZURE_OPENAI_ENDPOINT,
        "deploymentId": EMBEDDING_MODEL,
        "modelName": EMBEDDING_MODEL_NAME,
        "dimensions": EMBEDDING_DIMENSIONS,
        "inputs": [{"name": "text", "source": "/document/pages/*"}],
        "outputs": [{"name": "embedding", "targetName": "content_vector"}],
    }

    if AZURE_OPENAI_API_KEY:
        embedding_skill["apiKey"] = AZURE_OPENAI_API_KEY

    return {
        "name": SKILLSET_NAME,
        "description": "Chunking nativo con Azure AI Search Text Split y embeddings en Azure OpenAI.",
        "skills": [
            split_skill,
            embedding_skill,
        ],
        "indexProjections": {
            "selectors": [
                {
                    "targetIndexName": AZURE_SEARCH_INDEX,
                    "parentKeyFieldName": "parent_id",
                    "sourceContext": "/document/pages/*",
                    "mappings": [
                        {"name": "content", "source": "/document/pages/*"},
                        {"name": "content_vector", "source": "/document/pages/*/content_vector"},
                        {"name": "title", "source": "/document/title"},
                        {"name": "source_url", "source": "/document/source_url"},
                        {"name": "category", "source": "/document/category"},
                        {"name": "mission", "source": "/document/mission"},
                    ],
                }
            ],
            "parameters": {"projectionMode": "skipIndexingParentDocuments"},
        },
    }


def indexer_definition() -> dict:
    return {
        "name": INDEXER_NAME,
        "dataSourceName": DATA_SOURCE_NAME,
        "targetIndexName": AZURE_SEARCH_INDEX,
        "skillsetName": SKILLSET_NAME,
        "parameters": {
            "configuration": {
                "dataToExtract": "contentAndMetadata",
                "parsingMode": "default",
                "indexedFileNameExtensions": ".pdf",
            }
        },
    }


def reset_assets(client: httpx.Client) -> None:
    if not RESET_SEARCH_ASSETS:
        return

    print("Recreando recursos de Azure AI Search...")
    for path in (
        resource_path("indexers", INDEXER_NAME),
        resource_path("skillsets", SKILLSET_NAME),
        resource_path("datasources", DATA_SOURCE_NAME),
        resource_path("indexes", AZURE_SEARCH_INDEX),
    ):
        request_search(client, "DELETE", path, ignore_404=True)


with httpx.Client(timeout=120.0, headers=auth_headers()) as search_admin_client:
    reset_assets(search_admin_client)

    print("Creando índice vectorial...")
    request_search(search_admin_client, "PUT", resource_path("indexes", AZURE_SEARCH_INDEX), json_body=index_definition())

    print("Creando data source hacia Azure Blob Storage...")
    request_search(search_admin_client, "PUT", resource_path("datasources", DATA_SOURCE_NAME), json_body=data_source_definition())

    print("Creando skillset con Text Split + Azure OpenAI Embeddings...")
    request_search(search_admin_client, "PUT", resource_path("skillsets", SKILLSET_NAME), json_body=skillset_definition())

    print("Creando indexer...")
    request_search(search_admin_client, "PUT", resource_path("indexers", INDEXER_NAME), json_body=indexer_definition())

print("Recursos de Azure AI Search listos.")


## 7. Ejecutar el indexer y preparar el cliente de búsqueda

El indexer procesa los blobs, aplica el skillset y proyecta los chunks en el índice. Al terminar, el mismo índice queda listo para búsqueda híbrida y Semantic Ranker.


In [ ]:
import time

from azure.search.documents import SearchClient


def run_indexer(client: httpx.Client) -> None:
    response = client.post(search_url(f"{resource_path('indexers', INDEXER_NAME)}/search.run"))
    if response.status_code == 409:
        print("El indexer ya está en ejecución.")
        return
    if response.status_code not in (200, 202, 204):
        raise RuntimeError(
            f"POST {resource_path('indexers', INDEXER_NAME)}/search.run "
            f"falló con HTTP {response.status_code}:\n"
            f"{response.text}"
        )


def wait_for_indexer(client: httpx.Client) -> None:
    if not WAIT_FOR_INDEXER:
        print("Indexer iniciado. Puedes revisar el estado en Azure Portal.")
        return

    print("Esperando a que termine el indexer...")
    for _ in range(30):
        response = request_search(client, "GET", f"{resource_path('indexers', INDEXER_NAME)}/search.status")
        status = response.json()
        last_result = status.get("lastResult") or {}
        indexer_status = last_result.get("status") or status.get("status")

        if indexer_status:
            processed = last_result.get("itemsProcessed", 0)
            failed = last_result.get("itemsFailed", 0)
            print(f"Estado: {indexer_status} | procesados={processed} | fallidos={failed}")

        if indexer_status in {"success", "transientFailure", "persistentFailure"}:
            errors = last_result.get("errors") or []
            warnings = last_result.get("warnings") or []
            for warning in warnings[:5]:
                print("Advertencia:", warning.get("message", warning))
            for error in errors[:5]:
                print("Error:", error.get("message", error))
            if indexer_status != "success":
                raise RuntimeError("El indexer terminó con errores. Revisa el detalle anterior.")
            return

        time.sleep(10)

    print("El indexer sigue en ejecución. Revisa el progreso en Azure Portal.")


with httpx.Client(timeout=120.0, headers=auth_headers()) as search_admin_client:
    print("Ejecutando indexer...")
    run_indexer(search_admin_client)
    wait_for_indexer(search_admin_client)

search_client = SearchClient(
    endpoint=AZURE_SEARCH_ENDPOINT,
    index_name=AZURE_SEARCH_INDEX,
    credential=credential,
)

print("Cliente de búsqueda listo.")


## 8. Probar recuperación híbrida con Rerank (Semantic Ranker)

La consulta híbrida envía simultáneamente:

- `search_text`: coincidencia léxica BM25.
- `vector_queries`: similitud semántica.
Azure AI Search combina ambos rankings y aplica un paso de **Rerank (Semantic Ranker)** para reordenar los resultados utilizando modelos de lenguaje avanzados, priorizando la relevancia semántica.


In [27]:
from azure.search.documents.models import VectorizedQuery


async def retrieve(query: str, top_k: int = 5, use_rerank: bool = True) -> list[dict]:
    query_vector = (await embed_texts([query]))[0]

    vector_query = VectorizedQuery(
        vector=query_vector,
        k_nearest_neighbors=top_k,
        fields="content_vector",
        kind="vector",
    )

    if use_rerank:
        try:
            # Intentamos usar búsqueda semántica (Rerank L2)
            results = search_client.search(
                search_text=query,
                vector_queries=[vector_query],
                select=["id", "title", "content", "source_url", "page_number"],
                top=top_k,
                query_type="semantic",
                semantic_configuration_name="rag-semantic-config",
            )
        except Exception as e:
            print(f"Advertencia: No se pudo realizar la búsqueda semántica (Rerank). Fallando al modo híbrido estándar: {e}")
            results = search_client.search(
                search_text=query,
                vector_queries=[vector_query],
                select=["id", "title", "content", "source_url", "page_number"],
                top=top_k,
            )
    else:
        results = search_client.search(
            search_text=query,
            vector_queries=[vector_query],
            select=["id", "title", "content", "source_url", "page_number"],
            top=top_k,
        )

    return [
        {
            "id": result["id"],
            "title": result["title"],
            "content": result["content"],
            "source_url": result["source_url"],
            "page_number": result.get("page_number"),
            "score": result.get("@search.score"),
        }
        for result in results
    ]


sample_query = "What is the Curiosity's mission?"
print("--- Recuperando con Rerank ---")
retrieved_con = await retrieve(sample_query, top_k=4, use_rerank=True)
for rank, item in enumerate(retrieved_con, start=1):
    location = item.get("page_number") or "N/A"
    print(f'[{rank}] {item["title"]}, ubicación {location}, score={item["score"]}')
    print(item["content"][:200], "\n")

print("\n--- Recuperando sin Rerank ---")
retrieved_sin = await retrieve(sample_query, top_k=4, use_rerank=False)
for rank, item in enumerate(retrieved_sin, start=1):
    location = item.get("page_number") or "N/A"
    print(f'[{rank}] {item["title"]}, ubicación {location}, score={item["score"]}')
    print(item["content"][:200], "\n")


--- Recuperando con Rerank ---
[1] Mars Science Laboratory Curiosity Landing Press Kit, página 49, score=0.025484349578619003
Mars Science Laboratory Landing 49 Press Kit Curiosity’s landing area and surrounding terrain at Gale Crater, looking toward the southeast • Among the exposures in the lower portion of Mount Sharp are 

[2] Mars Science Laboratory Curiosity Landing Press Kit, página 10, score=0.011363636702299118
Mars Science Laboratory Landing 10 Press Kit mersed in tar pits. Mars won’t have fossils of insects or mastodons; if Mars has had any life forms at all, they were likely microbes. Understanding what t 

[3] Mars Science Laboratory Curiosity Landing Press Kit, página 32, score=0.01819859817624092
surface operations, the rover Curiosity has multiple options available for receiving commands from mission controllers on Earth and for returning rover sci- ence and engineering information. Curiosity 

[4] Mars Science Laboratory Curiosity Landing Press Kit, página 47, score=0.

## 9. Convertir Azure AI Search en una herramienta del agente

La función devuelve únicamente los fragmentos recuperados. El modelo recibe instrucciones estrictas para:

- usar la herramienta antes de responder;
- no inventar información;
- citar título, página y URL;
- reconocer cuando la base documental no contiene la respuesta.


In [28]:
from typing import Annotated
from agent_framework import tool
from pydantic import Field


@tool(approval_mode="never_require")
async def buscar_en_base_documental_con_rerank(
    pregunta: Annotated[
        str,
        Field(description="Pregunta o consulta de búsqueda sobre las misiones a Marte.")
    ],
    top_k: Annotated[
        int,
        Field(description="Número de fragmentos a recuperar.", ge=1, le=8)
    ] = 5,
) -> str:
    """Busca evidencia relevante en el índice de Azure AI Search usando Rerank (Búsqueda Semántica L2)."""
    results = await retrieve(pregunta, top_k=top_k, use_rerank=True)
    if not results:
        return "NO_SE_ENCONTRO_EVIDENCIA"

    passages = []
    for i, item in enumerate(results, start=1):
        passages.append(
            f"FRAGMENTO {i}\n"
            f"Título: {item['title']}\n"
            f"Ubicación: {item.get('page_number') or 'N/A'}\n"
            f"Fuente: {item['source_url']}\n"
            f"Contenido: {item['content']}"
        )
    return "\n\n---\n\n".join(passages)


@tool(approval_mode="never_require")
async def buscar_en_base_documental_sin_rerank(
    pregunta: Annotated[
        str,
        Field(description="Pregunta o consulta de búsqueda sobre las misiones a Marte.")
    ],
    top_k: Annotated[
        int,
        Field(description="Número de fragmentos a recuperar.", ge=1, le=8)
    ] = 5,
) -> str:
    """Busca evidencia relevante en el índice usando búsqueda híbrida estándar (sin Rerank)."""
    results = await retrieve(pregunta, top_k=top_k, use_rerank=False)
    if not results:
        return "NO_SE_ENCONTRO_EVIDENCIA"

    passages = []
    for i, item in enumerate(results, start=1):
        passages.append(
            f"FRAGMENTO {i}\n"
            f"Título: {item['title']}\n"
            f"Ubicación: {item.get('page_number') or 'N/A'}\n"
            f"Fuente: {item['source_url']}\n"
            f"Contenido: {item['content']}"
        )
    return "\n\n---\n\n".join(passages)


## 10. Crear y ejecutar el agente RAG

In [29]:
from agent_framework.openai import OpenAIChatClient

chat_client = OpenAIChatClient(
    model=CHAT_MODEL,
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_version=None,
    credential=credential,
)

instructions = """
Eres un asistente RAG especializado en los documentos sobre misiones a Marte.

Reglas obligatorias:
1. Para cualquier pregunta factual sobre el corpus, llama primero a la herramienta de búsqueda disponible.
2. Responde solamente con información respaldada por los fragmentos recuperados.
3. Si la evidencia no es suficiente, indica claramente:
   "No encuentro evidencia suficiente en la base documental."
4. Incluye una sección final "Fuentes" con título, página y URL.
5. No inventes páginas, URLs, datos ni conclusiones.
6. Responde en el idioma de la pregunta.
7. Mantén la respuesta clara y apropiada para una demostración técnica.
"""

agent_con_rerank = chat_client.as_agent(
    name="MarsMissionsRAG_Rerank",
    instructions=instructions,
    tools=[buscar_en_base_documental_con_rerank],
)

agent_sin_rerank = chat_client.as_agent(
    name="MarsMissionsRAG_NoRerank",
    instructions=instructions,
    tools=[buscar_en_base_documental_sin_rerank],
)

question = "What is the Curiosity's mission?"

print("=== RESPUESTA CON RERANK (SEMANTIC RANKER) ===")
response_con = await agent_con_rerank.run(question)
print(response_con.text)

print("\n" + "=" * 60 + "\n")

print("=== RESPUESTA SIN RERANK (HÍBRIDO ESTÁNDAR) ===")
response_sin = await agent_sin_rerank.run(question)
print(response_sin.text)


=== RESPUESTA CON RERANK (SEMANTIC RANKER) ===
Curiosity's mission is to evaluate whether the Gale Crater area of Mars could have once supported life. Specifically, Curiosity investigates habitable environments by:

- Determining the nature and inventory of organic carbon compounds, searching for the chemical building blocks of life, and identifying features that may record the actions of biologically relevant processes.
- Characterizing the geology of its field site by investigating the chemical, isotopic, and mineralogical composition of surface and near-surface materials and interpreting the processes that formed rocks and soils.
- Analyzing environmental changes over time, including transitions from habitable to non-habitable conditions, particularly by studying the layers of rocks inside Gale Crater.
- Investigating planetary processes such as water cycles, modern environmental conditions, and preserving evidence of organics possibly derived from ancient microbial life.
- Providin

## 11. Pruebas recomendadas para la presentación

Ejecuta una pregunta de cada tipo:

1. **Recuperable y semántica**  
   `¿Cuál fue el objetivo principal de la misión Curiosity en Marte?`

2. **Pregunta de síntesis**  
   `Resume los descubrimientos clave sobre la posible presencia de agua en Marte.`

3. **Sin palabras exactas del documento**  
   `¿Qué indicios hay de que el planeta rojo pudo haber sido habitable en el pasado?`

4. **Fuera del corpus / prueba de grounding**  
   `¿Cuál es el precio actual de Bitcoin?`  
   El agente debe reconocer que no existe evidencia suficiente.

5. **Transparencia del RAG**  
   Primero muestra los chunks recuperados con `retrieve()`, y después la respuesta final del agente.


In [20]:
demo_questions = [
    "¿Cuál fue el objetivo principal de la misión Curiosity en Marte?",
    "Resume los descubrimientos clave sobre la posible presencia de agua en Marte.",
    "¿Qué indicios hay de que el planeta rojo pudo haber sido habitable en el pasado?",
    "¿Cuál es el precio actual de Bitcoin?",
]

#Descomenta para ejecutar la batería completa.
for q in demo_questions:
    print("\n" + "=" * 100)
    print("PREGUNTA:", q)
    
    print("\n--- Con Rerank (Semantic Ranker) ---")
    answer_con = await agent_con_rerank.run(q)
    print(answer_con.text)
    
    print("\n--- Sin Rerank (Híbrido Estándar) ---")
    answer_sin = await agent_sin_rerank.run(q)
    print(answer_sin.text)


## 12. Qué explicar durante la demo

### Flujo de ingestión

`PDF → Blob Storage → Azure AI Search Indexer → Text Split Skill → AzureOpenAIEmbeddingSkill → index projections → índice de chunks`

### Flujo de consulta

`pregunta → embedding → búsqueda híbrida → Rerank (Semantic Ranker) → top-k chunks → herramienta → agente → respuesta citada`

### Diferencia entre RAG y entrenamiento

Los documentos no reentrenan el modelo. Se recuperan en tiempo de consulta y se incluyen como contexto.

### Por qué usar herramientas nativas de Azure

La demo evita implementar parsing, chunking y vectorización documental a mano. Azure AI Search administra el pipeline de ingestión y permite explicar claramente dónde ocurre cada parte: extracción, chunking, embedding, proyección e indexación.

### Por qué usar un agente

Un pipeline RAG tradicional llama siempre al buscador. En esta demo, Agent Framework registra la recuperación como una herramienta y administra el ciclo de tool calling. Esto permite agregar más herramientas posteriormente, por ejemplo SQL, APIs empresariales o acciones.

### Limitaciones que conviene mencionar

- La calidad depende del parsing, chunking, embeddings y ranking.
- Los PDFs escaneados requerirían OCR o Azure Document Intelligence.
- Para producción se necesitan evaluación, observabilidad, seguridad, filtros por permisos y controles contra prompt injection.
- `AzureCliCredential` es conveniente para desarrollo; en producción se recomienda una identidad administrada específica.


## 13. Limpieza opcional

La siguiente celda elimina el índice para evitar dejar recursos de datos. Los recursos de Azure deben eliminarse desde el grupo de recursos cuando termine la demo.


In [ ]:
# PELIGRO: descomenta únicamente cuando quieras borrar los recursos de indexación.
# with httpx.Client(timeout=120.0, headers=auth_headers()) as search_admin_client:
#     for path in (
#         resource_path("indexers", INDEXER_NAME),
#         resource_path("skillsets", SKILLSET_NAME),
#         resource_path("datasources", DATA_SOURCE_NAME),
#         resource_path("indexes", AZURE_SEARCH_INDEX),
#     ):
#         request_search(search_admin_client, "DELETE", path, ignore_404=True)
# print("Recursos de Azure AI Search eliminados.")


## Referencias técnicas

- Microsoft Agent Framework: https://learn.microsoft.com/agent-framework/
- Provider Azure OpenAI / OpenAI para Agent Framework: https://learn.microsoft.com/agent-framework/agents/providers/openai
- Azure AI Search Text Split Skill: https://learn.microsoft.com/azure/search/cognitive-search-skill-textsplit
- Integrated vectorization en Azure AI Search: https://learn.microsoft.com/azure/search/vector-search-integrated-vectorization
- Index projections para RAG: https://learn.microsoft.com/azure/search/search-how-to-define-index-projections
- Blob indexers en Azure AI Search: https://learn.microsoft.com/azure/search/search-how-to-index-azure-blob-storage
